## 1. First look at the raw data

Load with no interpretation, to find how missing or unknown values are encoded before deciding how to parse them.


In [22]:
from pathlib import Path

import numpy as np
import pandas as pd

DATA_DIR = Path("../data/raw")  # notebook runs from notebooks/

df_peek = pd.read_csv(
    DATA_DIR / "diabetic_data.csv",
    keep_default_na=False,  # keep "NA", "None", "" etc. as raw text
    low_memory=False,       # infer each column's dtype from the whole file, not per chunk
)

print(df_peek.shape)
print(df_peek.info())
df_peek.head()


(101766, 50)
<class 'pandas.DataFrame'>
RangeIndex: 101766 entries, 0 to 101765
Data columns (total 50 columns):
 #   Column                    Non-Null Count   Dtype
---  ------                    --------------   -----
 0   encounter_id              101766 non-null  int64
 1   patient_nbr               101766 non-null  int64
 2   race                      101766 non-null  str  
 3   gender                    101766 non-null  str  
 4   age                       101766 non-null  str  
 5   weight                    101766 non-null  str  
 6   admission_type_id         101766 non-null  int64
 7   discharge_disposition_id  101766 non-null  int64
 8   admission_source_id       101766 non-null  int64
 9   time_in_hospital          101766 non-null  int64
 10  payer_code                101766 non-null  str  
 11  medical_specialty         101766 non-null  str  
 12  num_lab_procedures        101766 non-null  int64
 13  num_procedures            101766 non-null  int64
 14  num_medications   

,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),?,1,1,7,2,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),?,1,1,7,1,...,No,Steady,No,No,No,No,No,Ch,Yes,NO


### 1.1 Cardinality of the text columns

How many distinct values each text column has. This decides how to inspect it: columns with few
values can be read in full, columns with many values must be scanned.


In [19]:
# Text columns only (pandas 3 gives them the "str" dtype)
text_cols = df_peek.select_dtypes(include="str").columns

# Distinct values per column, smallest first
cardinality = df_peek[text_cols].nunique().sort_values()
cardinality


citoglipton                   1
examide                       1
acetohexamide                 2
troglitazone                  2
diabetesMed                   2
glipizide-metformin           2
tolbutamide                   2
metformin-rosiglitazone       2
metformin-pioglitazone        2
change                        2
glimepiride-pioglitazone      2
tolazamide                    3
gender                        3
readmitted                    3
glyburide-metformin           4
insulin                       4
miglitol                      4
acarbose                      4
rosiglitazone                 4
pioglitazone                  4
glyburide                     4
glimepiride                   4
chlorpropamide                4
nateglinide                   4
repaglinide                   4
metformin                     4
A1Cresult                     4
max_glu_serum                 4
glipizide                     4
race                          6
weight                       10
age     

### 1.2 Values of each text column

Columns with up to 30 distinct values (what fits in one readable line) are listed in full, so no
value can hide. For the others only the rarest values are shown: odd codes and typos live in the tail.

In [20]:
MAX_LISTED = 30  # readability limit: above this, a column is sampled instead of listed

for col, n_unique in cardinality.items():
    counts = df_peek[col].value_counts()  # each value with its frequency, most frequent first
    if n_unique <= MAX_LISTED:
        print(f"{col}: {counts.to_dict()}")
    else:
        # The tail holds the rarest values, where unusual codes show up
        print(f"{col} ({n_unique} unique), rarest: {counts.tail(10).to_dict()}")


citoglipton: {'No': 101766}
examide: {'No': 101766}
acetohexamide: {'No': 101765, 'Steady': 1}
troglitazone: {'No': 101763, 'Steady': 3}
diabetesMed: {'Yes': 78363, 'No': 23403}
glipizide-metformin: {'No': 101753, 'Steady': 13}
tolbutamide: {'No': 101743, 'Steady': 23}
metformin-rosiglitazone: {'No': 101764, 'Steady': 2}
metformin-pioglitazone: {'No': 101765, 'Steady': 1}
change: {'No': 54755, 'Ch': 47011}
glimepiride-pioglitazone: {'No': 101765, 'Steady': 1}
tolazamide: {'No': 101727, 'Steady': 38, 'Up': 1}
gender: {'Female': 54708, 'Male': 47055, 'Unknown/Invalid': 3}
readmitted: {'NO': 54864, '>30': 35545, '<30': 11357}
glyburide-metformin: {'No': 101060, 'Steady': 692, 'Up': 8, 'Down': 6}
insulin: {'No': 47383, 'Steady': 30849, 'Down': 12218, 'Up': 11316}
miglitol: {'No': 101728, 'Steady': 31, 'Down': 5, 'Up': 2}
acarbose: {'No': 101458, 'Steady': 295, 'Up': 10, 'Down': 3}
rosiglitazone: {'No': 95401, 'Steady': 6100, 'Up': 178, 'Down': 87}
pioglitazone: {'No': 94438, 'Steady': 6976

### 1.3 Scan for missing-value placeholders

Missing values are often written as text placeholders instead of empty cells. Every text column is
scanned for the usual ones: empty or punctuation-only strings (`?`, `-`) and words like `NA`, `null`,
`none`, `unknown`. Each hit is a candidate: whether it really means "missing" is decided case by case.


In [23]:
# Matches the whole value: empty/punctuation-only strings, or a common placeholder word
SENTINEL_PATTERN = r"^\W*$|^(na|n/a|nan|null|none|missing|not available|unknown.*)$"

for col in text_cols:
    # Strip spaces and lowercase before matching, so " NULL " and "Null" are caught too
    normalised = df_peek[col].str.strip().str.lower()
    # Count the matching values as they are written in the file
    hits = df_peek.loc[normalised.str.fullmatch(SENTINEL_PATTERN), col].value_counts()
    if not hits.empty:
        print(f"{col}: {hits.to_dict()}")


race: {'?': 2273}
gender: {'Unknown/Invalid': 3}
weight: {'?': 98569}
payer_code: {'?': 40256}
medical_specialty: {'?': 49949}
diag_1: {'?': 21}
diag_2: {'?': 358}
diag_3: {'?': 1423}
max_glu_serum: {'None': 96420}
A1Cresult: {'None': 84748}


### 1.4 Coded IDs: unknown values in `IDS_mapping.csv`

`admission_type_id`, `discharge_disposition_id` and `admission_source_id` are integer codes, so the
text scan above cannot see their placeholders. Their meaning is in `IDS_mapping.csv`: here we find the
codes whose description means "unknown" and count how often each one is actually used in the data.


In [24]:
# The file stacks three code tables, each starting with a "<column>,description" header row
ids_raw = pd.read_csv(
    DATA_DIR / "IDS_mapping.csv",
    header=None,
    names=["code", "description"],
    dtype=str,
    keep_default_na=False,
    skipinitialspace=True,  # strip the space after the comma, e.g. "9, Not Available"
)

# Drop the "," separator lines between the tables
ids_raw = ids_raw[ids_raw["code"] != ""]

# Tag each code with the column it belongs to: take the header name and fill it down
is_header = ids_raw["code"].str.endswith("_id")
ids_raw["column"] = ids_raw["code"].where(is_header).ffill()
ids_map = ids_raw[~is_header].astype({"code": int})[["column", "code", "description"]]

# "Not Mapped" added after reading the file: the generic text pattern did not include it
ID_SENTINEL_PATTERN = r"^(null|not available|not mapped|unknown.*)$"
is_unknown = ids_map["description"].str.strip().str.lower().str.fullmatch(ID_SENTINEL_PATTERN)
unknown_ids = ids_map[is_unknown].copy()

# How often each "unknown" code actually appears in the data
unknown_ids["n_rows"] = [
    (df_peek[col] == code).sum() for col, code in zip(unknown_ids["column"], unknown_ids["code"])
]
unknown_ids


,column,code,description,n_rows
5,admission_type_id,5,Not Available,4785
6,admission_type_id,6,NULL,5291
8,admission_type_id,8,Not Mapped,320
28,discharge_disposition_id,18,NULL,3691
35,discharge_disposition_id,25,Not Mapped,989
36,discharge_disposition_id,26,Unknown/Invalid,0
51,admission_source_id,9,Not Available,125
57,admission_source_id,15,Not Available,0
58,admission_source_id,17,NULL,6781
61,admission_source_id,20,Not Mapped,161


### 1.5 Missing-value inventory

| Kind | Where | Handling |
| --- | --- | --- |
| `?` | `race`, `weight`, `payer_code`, `medical_specialty`, `diag_1/2/3` | read as `NaN`, decided per column in data prep |
| `Unknown/Invalid` | `gender` (3 rows) | rows dropped in scoping |
| Coded "unknown" IDs (`NULL`, `Not Available`, `Not Mapped`, `Unknown/Invalid` in `IDS_mapping.csv`) | `admission_type_id`, `discharge_disposition_id`, `admission_source_id` | merged into one "Unknown" category in data prep |
| `None` | `A1Cresult`, `max_glu_serum` | **not missing**: the test was not measured, kept as a category |


In [25]:
df_raw = pd.read_csv(
    DATA_DIR / "diabetic_data.csv",
    na_values=["?"],        # the dataset's only missing-value token
    keep_default_na=False,  # keep "None" (test not measured) as a category
    low_memory=False,
)

# Fails loudly if pandas ever turns "None" into NaN again
assert (df_raw["A1Cresult"] == "None").any()

# Missing values per column, as count and percentage
n_missing = df_raw.isna().sum()
missing = pd.DataFrame({"n_missing": n_missing, "pct": (n_missing / len(df_raw) * 100).round(1)})
missing[missing["n_missing"] > 0].sort_values("n_missing", ascending=False)


,n_missing,pct
weight,98569,96.9
medical_specialty,49949,49.1
payer_code,40256,39.6
race,2273,2.2
diag_3,1423,1.4
diag_2,358,0.4
diag_1,21,0.0
